### hmm - hidden markov model

In [1]:
from collections import defaultdict, Counter
import pandas as pd
import numpy as np

s1 = [
    [(" ", "marry", "jane", "can", "see", "will", " "), ("<s>", "N", "N", "M", "V", "N", "<e>")],
    [(" ", "will", "jane", "spot", "marry", " "), ("<s>", "M", "N", "V", "N", "<e>")],
    [(" ", "spot", "will", "see", "marry", " "), ("<s>", "N", "M", "V", "N", "<e>")],
    [(" ", "marry", "will", "pat", "spot", " "), ("<s>", "N", "M", "V", "N", "<e>")]
]

vocab = ["maary", "jane", "can", "see", "will", "pat", "spot"]
len_vocab = len(vocab)

In [2]:
# all words
# all tags
# emission count 
# transition counts

In [3]:
# all words and all tags

all_tags = []
all_words = []

for words, tags in s1:
    all_tags.extend(tags)
    all_words.extend(words)


print("words", all_words)
print("tags", all_tags)

words [' ', 'marry', 'jane', 'can', 'see', 'will', ' ', ' ', 'will', 'jane', 'spot', 'marry', ' ', ' ', 'spot', 'will', 'see', 'marry', ' ', ' ', 'marry', 'will', 'pat', 'spot', ' ']
tags ['<s>', 'N', 'N', 'M', 'V', 'N', '<e>', '<s>', 'M', 'N', 'V', 'N', '<e>', '<s>', 'N', 'M', 'V', 'N', '<e>', '<s>', 'N', 'M', 'V', 'N', '<e>']


In [4]:
emission_counts = defaultdict(lambda: defaultdict(int))
transition_counts = defaultdict(lambda: defaultdict(int))

for w, t in zip(all_words, all_tags):
    emission_counts[t][w] += 1

for _, tags in s1:
    for i in range(len(tags) - 1):
        transition_counts[tags[i]][tags[i+1]] += 1

In [5]:
emission_counts

defaultdict(<function __main__.<lambda>()>,
            {'<s>': defaultdict(int, {' ': 4}),
             'N': defaultdict(int,
                         {'marry': 4, 'jane': 2, 'will': 1, 'spot': 2}),
             'M': defaultdict(int, {'can': 1, 'will': 3}),
             'V': defaultdict(int, {'see': 2, 'spot': 1, 'pat': 1}),
             '<e>': defaultdict(int, {' ': 4})})

In [6]:
transition_counts

defaultdict(<function __main__.<lambda>()>,
            {'<s>': defaultdict(int, {'N': 3, 'M': 1}),
             'N': defaultdict(int, {'N': 1, 'M': 3, '<e>': 4, 'V': 1}),
             'M': defaultdict(int, {'V': 3, 'N': 1}),
             'V': defaultdict(int, {'N': 4})})

In [12]:
emission_prob = defaultdict(lambda: defaultdict(int))

for tag in emission_counts:
    total = sum(emission_counts[tag].values())
    if tag != "<s>" and tag != "<e>":
        for word in emission_counts[tag]:
            emission_prob[tag][word] = emission_counts[tag][word] / total

emission_prob

defaultdict(<function __main__.<lambda>()>,
            {'N': defaultdict(int,
                         {'marry': 0.4444444444444444,
                          'jane': 0.2222222222222222,
                          'will': 0.1111111111111111,
                          'spot': 0.2222222222222222}),
             'M': defaultdict(int, {'can': 0.25, 'will': 0.75}),
             'V': defaultdict(int, {'see': 0.5, 'spot': 0.25, 'pat': 0.25})})

In [15]:
transition_prob = defaultdict(lambda: defaultdict(int))

for tag in emission_counts:
    total = sum(transition_counts[tag].values())
    for next_tag in transition_counts[tag]:
        transition_prob[tag][next_tag] = transition_counts[tag][next_tag] / total

transition_prob

defaultdict(<function __main__.<lambda>()>,
            {'<s>': defaultdict(int, {'N': 0.75, 'M': 0.25}),
             'N': defaultdict(int,
                         {'N': 0.1111111111111111,
                          'M': 0.3333333333333333,
                          '<e>': 0.4444444444444444,
                          'V': 0.1111111111111111}),
             'M': defaultdict(int, {'V': 0.75, 'N': 0.25}),
             'V': defaultdict(int, {'N': 1.0})})

In [36]:
import itertools

states = ["N", "V", "M"]
sentence = ["will", "can", "spot", "marry"]

best_prob = 0
best_tag = None

for tag in itertools.product(states, repeat=len(sentence)):

    prob = 1.0

    # Start Transition
    prob *= transition_prob["<s>"][tag[0]]

    # First Emission
    prob *= emission_prob[tag[0]][sentence[0]]

    if prob == 0:
        continue

    for i in range(1, len(sentence)):

        prob *= transition_prob[tag[i-1]][tag[i]]
        prob *= emission_prob[tag[i]][sentence[i]]

        if prob == 0:
            break

    prob *= transition_prob[tag[-1]]["<e>"]

    # skips 0 for end one
    if prob == 0:
        continue
    
    if prob > best_prob:
        best_prob = prob
        best_tag = tag
        print("Tag", tag, "Probability", prob)

print("\nBest Tag", best_tag)
print("Best Probability", best_prob)

Tag ('N', 'M', 'N', 'N') Probability 8.467543904215141e-06
Tag ('N', 'M', 'V', 'N') Probability 0.00025720164609053495

Best Tag ('N', 'M', 'V', 'N')
Best Probability 0.00025720164609053495


In [22]:
emission_df = pd.DataFrame(emission_prob)
emission_df = emission_df.replace(np.NaN, '-')

transition_df = pd.DataFrame(transition_prob)
transition_df = transition_df.replace(np.NaN, '-')

In [23]:
emission_df

,N,M,V
marry,0.444444,-,-
jane,0.222222,-,-
will,0.111111,0.75,-
spot,0.222222,-,0.25
can,-,0.25,-
see,-,-,0.5
pat,-,-,0.25


In [24]:
trans_df

,<s>,N,M,V
N,0.75,0.111111,0.25,1.0
M,0.25,0.333333,-,-
<e>,-,0.444444,-,-
V,-,0.111111,0.75,-
